# Exploration des données SMART Backblaze — janvier 2026
Partie 2 (Serveurs) — ticket SCRUM-14. Objectif : comprendre les données, choisir le modèle de disque et justifier les attributs SMART retenus.

## 1. Découverte d'un jour de données

In [1]:
import glob, os
import pandas as pd

DATA_DIR = os.path.expanduser("~/data/backblaze")
fichiers = sorted(glob.glob(f"{DATA_DIR}/**/2026-01-*.csv", recursive=True))
print(len(fichiers), "fichiers")
print(fichiers[:2])

31 fichiers
['/home/imane/data/backblaze/data_Q1_2026/2026-01-01.csv', '/home/imane/data/backblaze/data_Q1_2026/2026-01-02.csv']


In [2]:
jour = pd.read_csv(fichiers[0])
print(jour.shape)
jour.head()

(338760, 197)


,date,serial_number,model,capacity_bytes,failure,datacenter,cluster_id,vault_id,pod_id,pod_slot_num,...,smart_250_normalized,smart_250_raw,smart_251_normalized,smart_251_raw,smart_252_normalized,smart_252_raw,smart_254_normalized,smart_254_raw,smart_255_normalized,smart_255_raw
0,2026-01-01,000a43e7dee60010,DELLBOSS VD,480036847616,0,ams5,31,2017,5,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-01-01,00ac486b9bec0010,DELLBOSS VD,480036847616,0,ams5,31,2013,16,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-01-01,01b776caeb6a0010,DELLBOSS VD,480036847616,0,ams5,31,2014,11,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-01-01,02308a3415be0010,DELLBOSS VD,480036847616,0,ams5,31,2007,4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-01-01,0394cd7bc5d00010,DELLBOSS VD,480036847616,0,ams5,31,2017,4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
[c for c in jour.columns if not c.startswith("smart")]

['date',
 'serial_number',
 'model',
 'capacity_bytes',
 'failure',
 'datacenter',
 'cluster_id',
 'vault_id',
 'pod_id',
 'pod_slot_num',
 'is_legacy_format']

In [4]:
SMART = ["smart_5_raw", "smart_9_raw", "smart_187_raw",
         "smart_194_raw", "smart_197_raw", "smart_198_raw"]

print("Pannes ce jour :", jour["failure"].sum())
print("\nPart de valeurs manquantes :")
print(jour[SMART].isna().mean().round(3))

Pannes ce jour : 1

Part de valeurs manquantes :
smart_5_raw      0.007
smart_9_raw      0.001
smart_187_raw    0.664
smart_194_raw    0.001
smart_197_raw    0.025
smart_198_raw    0.008
dtype: float64


## 2. Chargement du mois de janvier 2026

In [5]:
COLONNES = ["date", "serial_number", "model", "capacity_bytes", "failure"] + SMART
TYPES = {c: "float32" for c in SMART}
TYPES["failure"] = "int8"

morceaux = []
for i, f in enumerate(fichiers, start=1):
    morceaux.append(pd.read_csv(f, usecols=COLONNES, dtype=TYPES))
    print(f"{i}/{len(fichiers)} chargé", end="\r")

df = pd.concat(morceaux, ignore_index=True)
del morceaux

df["date"] = pd.to_datetime(df["date"])
df["model"] = df["model"].astype("category")
df["serial_number"] = df["serial_number"].astype("category")

print("\nTaille :", df.shape)
df.info(memory_usage="deep")

31/31 chargé
Taille : (10463270, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10463270 entries, 0 to 10463269
Data columns (total 11 columns):
 #   Column          Dtype         
---  ------          -----         
 0   date            datetime64[ns]
 1   serial_number   category      
 2   model           category      
 3   capacity_bytes  int64         
 4   failure         int8          
 5   smart_5_raw     float32       
 6   smart_9_raw     float32       
 7   smart_187_raw   float32       
 8   smart_194_raw   float32       
 9   smart_197_raw   float32       
 10  smart_198_raw   float32       
dtypes: category(2), datetime64[ns](1), float32(6), int64(1), int8(1)
memory usage: 486.1 MB


In [6]:
SORTIE = os.path.join(DATA_DIR, "janvier_2026_smart.parquet")
df.to_parquet(SORTIE, index=False)
print("Enregistré :", SORTIE, "-", round(os.path.getsize(SORTIE) / 1e6), "Mo")

Enregistré : /home/imane/data/backblaze/janvier_2026_smart.parquet - 77 Mo


## 3. Statistiques globales et par modèle de disque

In [7]:
print("Disques différents :", df["serial_number"].nunique())
print("Pannes sur le mois  :", int(df["failure"].sum()))
print("Modèles de disques  :", df["model"].nunique())

Disques différents : 341514
Pannes sur le mois  : 306
Modèles de disques  : 79


In [8]:
par_modele = df.groupby("model", observed=True).agg(
    disques=("serial_number", "nunique"),
    pannes=("failure", "sum"),
    capacite_To=("capacity_bytes", "max"),
    manque_187=("smart_187_raw", lambda s: s.isna().mean()),
)
par_modele["pannes"] = par_modele["pannes"].astype("int64")   # évite le dépassement de capacité de int8
par_modele["capacite_To"] = (par_modele["capacite_To"] / 1e12).round(1)
par_modele["taux_panne_%"] = (100 * par_modele["pannes"] / par_modele["disques"]).round(3)
par_modele["manque_187"] = par_modele["manque_187"].round(2)

par_modele.sort_values("disques", ascending=False).head(15)

,disques,pannes,capacite_To,manque_187,taux_panne_%
model,,,,,
WDC WUH722222ALE6L4,44401,5,22.0,1.0,0.011
TOSHIBA MG08ACA16TA,39680,30,16.0,1.0,0.076
TOSHIBA MG07ACA14TA,37258,33,14.0,1.0,0.089
ST16000NM001G,34445,12,16.0,0.0,0.035
WDC WUH721816ALE6L4,26316,11,16.0,1.0,0.042
ST12000NM0008,18646,49,12.0,0.0,0.263
TOSHIBA MG10ACA20TE,18014,12,20.0,1.0,0.067
HGST HUH721212ALE604,13267,23,12.0,1.0,0.173
ST8000NM0055,13220,7,8.0,0.0,0.053


In [9]:
par_modele.sort_values("pannes", ascending=False).head(10)

,disques,pannes,capacite_To,manque_187,taux_panne_%
model,,,,,
ST12000NM0008,18646,49,12.0,0.0,0.263
TOSHIBA MG07ACA14TA,37258,33,14.0,1.0,0.089
ST24000NM002H,9560,31,24.0,0.0,0.324
TOSHIBA MG08ACA16TA,39680,30,16.0,1.0,0.076
HGST HUH721212ALE604,13267,23,12.0,1.0,0.173
HGST HUH721212ALN604,9771,21,12.0,1.0,0.215
ST12000NM001G,13215,13,12.0,0.0,0.098
ST16000NM001G,34445,12,16.0,0.0,0.035
TOSHIBA MG10ACA20TE,18014,12,20.0,1.0,0.067


## 4. Disques en panne vs disques sains (ST12000NM0008)

In [10]:
import numpy as np

m = df[df["model"] == "ST12000NM0008"].copy()
m[SMART] = m[SMART].astype("float64")

series_panne = m.loc[m["failure"] == 1, "serial_number"].unique()
m["groupe"] = np.where(m["serial_number"].isin(series_panne), "panne", "sain")

dernier = m.sort_values("date").groupby("serial_number", observed=True).tail(1)
print(dernier["groupe"].value_counts())

groupe
sain     18597
panne       49
Name: count, dtype: int64


In [11]:
dernier.groupby("groupe")[["smart_9_raw", "smart_194_raw"]].median().round(0)

,smart_9_raw,smart_194_raw
groupe,,
panne,53148.0,31.0
sain,50985.0,32.0


In [12]:
COMPTEURS = ["smart_5_raw", "smart_187_raw", "smart_197_raw", "smart_198_raw"]

pct_non_nul = dernier.groupby("groupe")[COMPTEURS].apply(lambda g: (g > 0).mean() * 100).round(1)
pct_non_nul

,smart_5_raw,smart_187_raw,smart_197_raw,smart_198_raw
groupe,,,,
panne,71.4,85.7,51.0,51.0
sain,7.0,4.2,0.9,0.9


## 5. Conclusion

**Modèle de disque retenu : ST12000NM0008** (Seagate, 12 To) — 18 646 disques, 49 pannes en janvier 2026. Seuls les Seagate renseignent le SMART 187.

**Attributs SMART retenus (6) :**

| Attribut | Disques en panne | Disques sains | Rôle |
|---|---|---|---|
| smart_5_raw (secteurs réalloués) | 71,4 % | 7,0 % | compteur d'erreurs |
| smart_187_raw (erreurs non corrigibles) | 85,7 % | 4,2 % | compteur d'erreurs |
| smart_197_raw (secteurs en attente) | 51,0 % | 0,9 % | compteur d'erreurs |
| smart_198_raw (secteurs non corrigibles) | 51,0 % | 0,9 % | compteur d'erreurs (corrélé à 197) |
| smart_9_raw (heures) | médiane 53 148 h | 50 985 h | contexte |
| smart_194_raw (température) | médiane 31 °C | 32 °C | contexte |

*(% = part des disques avec au moins une erreur, au dernier relevé.)*

**Points d'attention pour la suite :**
- Déséquilibre très fort : environ 1 disque en panne pour 380 sains → métriques recall / précision / F1, pas l'accuracy.
- 49 pannes par mois ne suffisent pas pour entraîner M1 → utiliser plusieurs mois de données.
- Aucun attribut n'est suffisant seul → le modèle doit combiner les attributs et leur évolution sur 7 jours.

## 6. Vérification des données nettoyées (6 mois)

In [3]:
import os
import pandas as pd

propre = pd.read_parquet(os.path.expanduser("~/data/backblaze/clean"))
print(propre.shape)

pannes = propre[propre["label_7j"] == 1].groupby("device_id", observed=True).size()
print("Disques avec au moins une ligne label_7j = 1 :", len(pannes))
print(pannes.value_counts().sort_index())

(3269213, 11)
Disques avec au moins une ligne label_7j = 1 : 241
1     29
2      8
3     13
4      6
5     10
6     23
7    152
Name: count, dtype: int64
